In [1]:
import os
import urllib3
import urllib.request
# import torch

filename = './data/verdict.txt'
url = 'https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt'
if not os.path.exists(filename) or not os.path.getsize(filename):
    urllib.request.urlretrieve(url,filename)

In [2]:
with open(filename) as f:
    raw_text = f.read()

In [3]:
raw_text

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [4]:
len(raw_text)

20479

#### tokenization with regex

In [9]:
import re
_txt = 'Hello world, this is a text.'
tokens = [i  for i in re.split('([.,\?\s\t])',_txt) if i.strip()]
print(tokens)

['Hello', 'world', ',', 'this', 'is', 'a', 'text', '.']


In [6]:
txt = 'Hello world; IS this -- a tst?'
tokens = [i.lower()  for i in re.split('([.,\"\'_\-:;\?\s\t])',txt) if i.strip()]
print(tokens)

['hello', 'world', ';', 'is', 'this', '-', '-', 'a', 'tst', '?']


In [7]:
preprocessed = [i.lower()  for i in re.split('([.,"_\-:;()\'?\s\t])',raw_text) if i.strip()]
print(preprocessed)

['i', 'had', 'always', 'thought', 'jack', 'gisburn', 'rather', 'a', 'cheap', 'genius', '-', '-', 'though', 'a', 'good', 'fellow', 'enough', '-', '-', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'riviera', '.', '(', 'though', 'i', 'rather', 'thought', 'it', 'would', 'have', 'been', 'rome', 'or', 'florence', '.', ')', '"', 'the', 'height', 'of', 'his', 'glory', '"', '-', '-', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'i', 'can', 'hear', 'mrs', '.', 'gideon', 'thwing', '-', '-', 'his', 'last', 'chicago', 'sitter', '-', '-', 'deploring', 'his', 'unaccountable', 'abdication', '.', '"', 'of', 'course', 'it', "'", 's', 'going', 'to', 'send', 'the', 'value', 'of', 'my', 'picture', "'", 'way', 'up', ';', 'but', 'i', 'don', "'", 't', 'thin

In [8]:
len(preprocessed)


4839

In [9]:
all_words = sorted(set(preprocessed))
all_words.extend(['<|unk|>','<|endoftext|>'])
vocab_size = len(all_words)

In [10]:
vocab = {token:integer for integer,token in enumerate(all_words)}


In [ ]:
vocab

#### Simple tokenizer

In [12]:
class tokenizeV1:
    def __init__(self,vocab):
        self.s_to_i = vocab
        self.i_to_s = {i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = [i.lower() for i in re.split('([.,"_\-:;()\'?\s\t])',text) if i.strip()]
        ids = [self.s_to_i[w] if w in self.s_to_i else self.s_to_i['<|unk|>'] for w in preprocessed]
        return ids
    
    def decode(self,ids):
        words = [self.i_to_s[i] for i in ids]
        return ' '.join(words)

In [13]:
tokenizer = tokenizeV1(vocab)

In [14]:
t = '''"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride. "The last but one," she corrected herself--'''

ids = tokenizer.encode(t)

recreated_text = tokenizer.decode(ids)

print(ids)
print(recreated_text)

[0, 509, 1, 802, 947, 530, 452, 685, 4, 1099, 524, 4, 0, 626, 6, 405, 803, 1077, 694, 737, 6, 0, 947, 530, 138, 667, 4, 0, 829, 197, 462, 5, 5]
" it ' s the last he painted , you know , " mrs . gisburn said with pardonable pride . " the last but one , " she corrected herself - -


In [15]:
t = 'das auto'

# t = '''"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride. "The last but one," she corrected herself--'''

ids = tokenizer.encode(t)

recreated_text = tokenizer.decode(ids)

print(ids)
print(recreated_text)

[1103, 1103]
<|unk|> <|unk|>


#### Byte Pair Encoding

In [10]:
import tiktoken
import importlib
importlib.reload(tiktoken)

<module 'tiktoken' from '/Users/venugopalbhatia/Documents/Build_LLMS_From_Scratch/.venv/lib/python3.10/site-packages/tiktoken/__init__.py'>

In [17]:
tiktoken.__version__

'0.11.0'

In [18]:
tokenizer = tiktoken.get_encoding("gpt2")

In [19]:
tokenizer.special_tokens_set

{'<|endoftext|>'}

In [20]:
tokenizer

<Encoding 'gpt2'>

In [21]:
tokens = tokenizer.encode("Hello World")
print(tokens)
tokenizer.decode(tokens)

[15496, 2159]


'Hello World'

In [22]:
tokenizer.special_tokens_set
text = (
    "jdc dkhj ksn jelly hello ajdbc <|endoftext|>"
)

tokenizer.encode(text, allowed_special=tokenizer.special_tokens_set)

[73,
 17896,
 288,
 14636,
 73,
 479,
 16184,
 27644,
 23748,
 257,
 73,
 9945,
 66,
 220,
 50256]

#### Data Sampling with a Sliding Window

In [23]:
# text itself acts as the labels, n tokens have to predict the n+1th token

with open('data/verdict.txt') as f:
    text = f.read()
    tokens = tokenizer.encode(text)


In [24]:
tokenizer.decode([tokens[2]])

'AD'

In [25]:
window = 4

x = tokens[:window]
y = tokens[1:window+1]

print(x)
print("    ",y)


for i in range(1,5):
    print(tokens[:i],"---->",tokens[i])
    print(tokenizer.decode(tokens[:i]),'---->',tokenizer.decode([tokens[i]]))


[40, 367, 2885, 1464]
     [367, 2885, 1464, 1807]
[40] ----> 367
I ---->  H
[40, 367] ----> 2885
I H ----> AD
[40, 367, 2885] ----> 1464
I HAD ---->  always
[40, 367, 2885, 1464] ----> 1807
I HAD always ---->  thought


In [4]:
import torch
from torch.utils.data import DataLoader,Dataset


In [27]:
torch.__version__

'2.8.0'

In [36]:
%%writefile createDatasetTensors.py

# Create a Class that takes the text and using a tokenizer gives input and target tensors
# Create a DataLoader Class that returns batches of Data for Training

class createDatasetTensors(Dataset):
    def __init__(self,text,context_window,stride,tokenizer):
        self.input = []
        self.target = []

        tokens = tokenizer.encode(text)
        l = len(tokens)

        for i in range(0,l-context_window,stride):
            ip = tokens[i:i+context_window]
            op = tokens[i+1:i+context_window+1]

            self.input.append(torch.tensor(ip))
            self.target.append(torch.tensor(op))
    def __len__(self):
         return len(self.input)
    
    def __getitem__(self,idx):
        return self.input[idx],self.target[idx]

def create_dataloader(context_window,stride,model="gpt2",batch_size=32,shuffle=True,drop_last=True,num_workers = 0):

        tokenizer = tiktoken.get_encoding(model)
        with open('data/verdict.txt','r') as f:
            text = f.read()
        
        dataset = createDatasetTensors(text,context_window,stride,tokenizer)
        print(len(dataset))
        dataloader = DataLoader(
            dataset,
            batch_size,
            shuffle = shuffle,
            drop_last=drop_last,
            num_workers = num_workers
        )
        print(len(dataloader))
        return dataloader,dataset









Writing createDatasetTensors.py


## Generate embeddings

In [24]:
# Creating token embeddings
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
dir(tokenizer)
tokenizer.n_vocab
num_emb = tokenizer.n_vocab
emb_dim = 256
context_window = 4
torch.manual_seed(123)
token_embedding_layer = torch.nn.Embedding(num_emb,emb_dim)
token_embedding_layer.weight

import createDatasetTensors
importlib.reload(createDatasetTensors)
dataloader,dataset = createDatasetTensors.create_dataloader(context_window=context_window,stride=1)
tokens = next(iter(dataloader))
# print(tokens)
token_embeddings = token_embedding_layer(tokens[0])
token_embeddings.shape

position_embedding_layer = torch.nn.Embedding(context_window,emb_dim)
position_embeddings = position_embedding_layer(torch.arange(context_window))
position_embeddings.shape

input_embeddings = token_embeddings+position_embeddings
input_embeddings.shape

5141
160


torch.Size([32, 4, 256])